# 周频与日频策略方向探索（2026-07-22）

## TL;DR

- 周频仍应是主攻，但下一步应从“固定 Top60 硬中性”改为“真正的四期限重训 + 可行性扩池的风险 challenger”。现有信号下，neutral Top60 的 size±0.75 约束在 2025 仅 24/52 周可行。Top500 才达到 52/52。
- 风格硬约束能消除暴露，却显著削弱已开发样本收益：2025 费后复合超额从 44.62% 降至 19.63%，再加 beta band 后为 15.34%。它适合 challenger，不适合直接替代 incumbent。
- 日频状态规则在两窗、三个期限中均降低成交额，但收益不稳定。10bp 下，2026 跨源窗 H3 的 stateful−stateless 为 +7.62 个百分点，HAC t=2.05。2025 复用窗 H3 则为 −7.17 个百分点。当前只支持继续冻结验证，不支持晋级。
- 日频结果是 `target_state_proxy`：离线 target 没有把实际 fills 反馈给下一次选仓。完整 account-state 事件循环仍是下一基础设施里程碑。

## Context and decision frame

研究资源仍按周频 60%、日频 30%、高风险 shadow 10% 分配。本 notebook 只回答两个机制问题：

1. 周频硬风格约束在当前候选池里是否可行，代价多大？
2. 固定 D 分数后，状态化选仓和 H1/H3/H5 错峰持有是否稳定降低换手并改善费后结果？

2025 与当前可见的 2026 数据均已参与研究判断，全部结果标记为开发/复用证据，不是 untouched OOS。

In [1]:
from pathlib import Path
import json
import os
import shutil
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(os.environ.get('RESEARCH_APPS_ROOT', Path.cwd())).resolve()
OUTPUT = Path(os.environ.get(
    'STRATEGY_DIRECTION_OUTPUT',
    PROJECT_ROOT / 'artifacts/strategy_direction_exploration_20260722',
)).resolve()
OUTPUT.mkdir(parents=True, exist_ok=True)
DAILY_ROOT = Path(os.environ.get(
    'STATEFUL_STAGGERED_OUTPUT',
    PROJECT_ROOT / 'artifacts/stateful_staggered_20260722',
)).resolve()
WEEKLY_LP = OUTPUT / 'weekly_lp_probe'
WEEKLY_REPRO = OUTPUT / 'weekly_reliability_repro'
WEEKLY_REPRO_SOURCE = Path(os.environ.get('WEEKLY_REPRO_SOURCE', WEEKLY_REPRO)).resolve()
WEEKLY_ARCHIVE = Path(os.environ.get(
    'WEEKLY_RELIABILITY_ARCHIVE',
    OUTPUT / 'weekly_reliability_archive',
)).resolve()
WEEKLY_REPRO.mkdir(parents=True, exist_ok=True)

required = [DAILY_ROOT, WEEKLY_LP, WEEKLY_ARCHIVE]
assert all(path.exists() for path in required), required
repro_input = WEEKLY_REPRO_SOURCE if WEEKLY_REPRO_SOURCE.exists() else WEEKLY_REPRO
assert repro_input.exists(), repro_input
def display_path(path):
    try:
        return path.relative_to(PROJECT_ROOT)
    except ValueError:
        return f'<external:{path.name}>'

print('output:', display_path(OUTPUT))
print('weekly reproducibility source:', display_path(repro_input))

output: artifacts/strategy_direction_exploration_20260722
weekly reproducibility source: artifacts/strategy_direction_exploration_20260722/weekly_reliability_repro


## Data and reproducibility

周频可信度研究从只读迁移挂载按显式路径完整复跑。新旧 CSV 的数值列逐文件比较。路径字段允许因挂载点改变，浮点字符串允许机器级末位差异。日频读取两套已冻结 D 分数与执行价格：2025 discovery 复用窗和 2026 cross-source 复用窗。

In [2]:
for source in repro_input.iterdir():
    if source.is_file() and source.suffix in {'.csv', '.json'}:
        destination = WEEKLY_REPRO / source.name
        if source.resolve() != destination.resolve():
            shutil.copy2(source, destination)

comparison_rows = []
for new_file in sorted(repro_input.glob('*.csv')):
    old_file = WEEKLY_ARCHIVE / new_file.name
    if not old_file.exists():
        continue
    new = pd.read_csv(new_file)
    old = pd.read_csv(old_file)
    numeric = new.select_dtypes(include='number').columns.intersection(
        old.select_dtypes(include='number').columns
    )
    max_abs = float((new[numeric] - old[numeric]).abs().max().max()) if len(numeric) else 0.0
    comparison_rows.append({
        'file': new_file.name,
        'rows': len(new),
        'exact_equal': bool(new.equals(old)),
        'numeric_equal_1e12': bool(new.shape == old.shape and max_abs <= 1e-12),
        'max_abs_numeric_delta': max_abs,
    })
weekly_repro_comparison = pd.DataFrame(comparison_rows)
weekly_repro_comparison.to_csv(OUTPUT / 'weekly_repro_comparison.csv', index=False)
print(weekly_repro_comparison.to_string(index=False))
print('numeric matches:', int(weekly_repro_comparison.numeric_equal_1e12.sum()), '/', len(weekly_repro_comparison))

                              file  rows  exact_equal  numeric_equal_1e12  max_abs_numeric_delta
          active_risk_exposure.csv    63         True                True                    0.0
                    cost_sweep.csv    36         True                True                    0.0
          data_quality_summary.csv    10         True                True                    0.0
              decay_cumulative.csv    64         True                True                    0.0
                decay_marginal.csv    64         True                True                    0.0
               direction_audit.csv     7         True                True                    0.0
                   edge_scales.csv   558         True                True                    0.0
 existing_strategy_sensitivity.csv     7         True                True                    0.0
          factor_concentration.csv    63         True                True                    0.0
              factor_inventory

## Weekly result: the signal is slow, but fixed Top60 is not a feasible neutral universe

现有 post-training 研究仍显示累计 RankIC 随期限增加，因此“四个增量期限分别重训”值得做。这里并未冒充已完成重训。新增 LP 机制审计把单票、行业、size 与 beta 约束放到组合层，并记录每周求解状态。

In [3]:
decay = pd.read_csv(WEEKLY_REPRO / 'decay_cumulative.csv')
ensemble_decay = decay.query("period == 'validation_2025' and signal == 'equal_ensemble'")[
    ['horizon_days', 'n_dates', 'rank_ic_mean', 'top20_excess_mean', 'top_bottom_mean']
]
print('2025 equal-ensemble cumulative horizons')
print(ensemble_decay.to_string(index=False))

feasibility = pd.read_csv(WEEKLY_LP / 'feasibility.csv')
feasibility_summary = (
    feasibility.query("score == 'neutral_equal6'")
    .groupby(['period', 'pool_n'], as_index=False)
    .agg(
        weeks=('date', 'size'),
        size075_ok=('size075_feasible', 'sum'),
        size075_beta050_ok=('size075_beta050_feasible', 'sum'),
    )
)
feasibility_summary['size075_rate'] = feasibility_summary.size075_ok / feasibility_summary.weeks
feasibility_summary['size075_beta050_rate'] = feasibility_summary.size075_beta050_ok / feasibility_summary.weeks
feasibility_summary.to_csv(OUTPUT / 'weekly_lp_feasibility_summary.csv', index=False)
print('\nneutral equal6 feasibility')
print(feasibility_summary.to_string(index=False))

weekly_performance = pd.read_csv(WEEKLY_LP / 'performance.csv')
weekly_performance.to_csv(OUTPUT / 'weekly_lp_performance.csv', index=False)
print('\nweekly constrained mechanism replay')
print(weekly_performance[['variant','period','n_periods','excess_compounded','excess_sharpe','avg_l1_turnover','avg_active_log_mv','avg_active_beta']].to_string(index=False))

2025 equal-ensemble cumulative horizons
 horizon_days  n_dates  rank_ic_mean  top20_excess_mean  top_bottom_mean
            1      243      0.073831           0.002241         0.011710
            5      243      0.118017           0.011488         0.042626
           10      243      0.139164           0.019045         0.060713
           20      243      0.158240           0.044194         0.104608

neutral equal6 feasibility
                      period  pool_n  weeks  size075_ok  size075_beta050_ok  size075_rate  size075_beta050_rate
limited_2026_to_industry_end      60      8           1                   1      0.125000              0.125000
limited_2026_to_industry_end     100      8           6                   5      0.750000              0.625000
limited_2026_to_industry_end     200      8           8                   8      1.000000              1.000000
limited_2026_to_industry_end     500      8           8                   8      1.000000              1.000000
limited

## Daily result: state consistently saves trading, but does not consistently preserve alpha

对每个 horizon，stateless 与 stateful 使用同一 D 分数、同一执行价格与同一成本。H3/H5 分别维护 3/5 个独立 sleeve。保留仓只成交目标权重差额。跨 horizon 的 stateful 比较是“状态规则 + 持有期”的联合政策，纯持有期效应只看 stateless H1/H3/H5。

In [4]:
source_labels = {
    'slow_volume_discovery_20251223': '2025复用窗',
    'slow_volume_cross_source_20260706': '2026跨源复用窗',
}
summary_parts, comparison_parts, diagnostic_parts = [], [], []
receipt_parts, coverage_parts = [], []
daily_reports = {}
for source_id, label in source_labels.items():
    source = DAILY_ROOT / source_id
    summary = pd.read_parquet(source / 'summaries.parquet').assign(source_id=source_id, window=label)
    comparison = pd.read_parquet(source / 'comparisons.parquet').assign(source_id=source_id, window=label)
    diagnostic = pd.read_parquet(source / 'execution_diagnostics.parquet').assign(source_id=source_id, window=label)
    receipts = pd.read_parquet(source / 'selection_receipts.parquet').assign(source_id=source_id, window=label)
    coverage = pd.read_parquet(source / 'target_coverage.parquet').assign(source_id=source_id, window=label)
    daily_reports[source_id] = json.loads((source / 'report.json').read_text())
    summary_parts.append(summary); comparison_parts.append(comparison)
    diagnostic_parts.append(diagnostic); receipt_parts.append(receipts); coverage_parts.append(coverage)

daily_summaries = pd.concat(summary_parts, ignore_index=True)
daily_comparisons = pd.concat(comparison_parts, ignore_index=True)
daily_diagnostics = pd.concat(diagnostic_parts, ignore_index=True)
daily_receipts = pd.concat(receipt_parts, ignore_index=True)
daily_coverage = pd.concat(coverage_parts, ignore_index=True)
daily_summaries.to_csv(OUTPUT / 'daily_all_cost_summaries.csv', index=False)
daily_comparisons.to_csv(OUTPUT / 'daily_all_cost_comparisons.csv', index=False)

daily_10 = daily_summaries.query('single_side_cost_bps == 10').copy()
comparison_10 = daily_comparisons.query('single_side_cost_bps == 10').copy()
daily_10 = daily_10.merge(
    comparison_10[['source_id','horizon','total_return_delta','mean_traded_notional_to_nav_delta','hac_t_stat','hac_p_value']],
    on=['source_id','horizon'], how='left', validate='many_to_one'
)
daily_10['cell_label'] = daily_10['window'] + ' H' + daily_10['horizon'].astype(str)
daily_10.to_csv(OUTPUT / 'daily_10bps_summary.csv', index=False)
comparison_10.to_csv(OUTPUT / 'daily_10bps_comparisons.csv', index=False)
print(daily_10[['window','arm','horizon','sessions','total_return','annualized_sharpe_zero_rf','mean_traded_notional_to_nav','mean_cash_weight']].to_string(index=False))
print('\nstateful minus stateless at 10bps')
print(comparison_10[['window','horizon','total_return_delta','mean_traded_notional_to_nav_delta','hac_t_stat','hac_p_value']].to_string(index=False))

   window       arm  horizon  sessions  total_return  annualized_sharpe_zero_rf  mean_traded_notional_to_nav  mean_cash_weight
  2025复用窗  stateful        1       257      0.188747                   0.964498                     0.519591          0.021232
  2025复用窗 stateless        1       257      0.185573                   0.911167                     0.555516          0.019870
  2025复用窗  stateful        3       257      0.223363                   1.305181                     0.214668          0.027300
  2025复用窗 stateless        3       257      0.295057                   1.510633                     0.238033          0.020249
  2025复用窗  stateful        5       257      0.215117                   1.360985                     0.143336          0.037089
  2025复用窗 stateless        5       257      0.246248                   1.402897                     0.161158          0.019958
2026跨源复用窗  stateful        1       125      0.107064                   0.927927                     0.864844   

## Validation, uncertainty, and robustness

周频 LP 的 22 项自动校验必须全部通过。日频每个 cell 必须终端清仓，合成容量不得约束任何 fill，权威交易日历不得有整日缺口。目标行情覆盖不足会显式记录。由于 selection state 来自 previous target  actual fills，未成交买单仍可能被下轮当作 incumbent。这限制了结论的账户真实性。

In [5]:
weekly_checks = pd.read_csv(WEEKLY_REPRO / 'validation_checks.csv')
lp_checks = pd.read_csv(WEEKLY_LP / 'validation.csv')
validation_rows = [
    {'area':'weekly_reproduction','check':'all_csv_numeric_outputs_match','observed':f"{int(weekly_repro_comparison.numeric_equal_1e12.sum())}/{len(weekly_repro_comparison)}",'status':'pass' if weekly_repro_comparison.numeric_equal_1e12.all() else 'fail'},
    {'area':'weekly_reliability','check':'no_failed_validation_checks','observed':int(weekly_checks.status.eq('fail').sum()),'status':'pass' if not weekly_checks.status.eq('fail').any() else 'fail'},
    {'area':'weekly_lp','check':'all_22_lp_checks_pass','observed':f"{int(lp_checks.status.eq('pass').sum())}/{len(lp_checks)}",'status':'pass' if lp_checks.status.eq('pass').all() else 'fail'},
    {'area':'daily','check':'all_60_execution_cells_terminal','observed':int(daily_summaries.terminal_complete.sum()),'status':'pass' if daily_summaries.terminal_complete.all() and len(daily_summaries)==60 else 'fail'},
    {'area':'daily','check':'synthetic_capacity_never_bound','observed':int(daily_diagnostics.capacity_bound_fills.sum()),'status':'pass' if daily_diagnostics.capacity_bound_fills.sum()==0 else 'fail'},
    {'area':'daily','check':'target_entry_price_coverage_at_least_99.9pct','observed':float(daily_coverage.entry_price_row_coverage.min()),'status':'pass' if daily_coverage.entry_price_row_coverage.min()>=0.999 else 'fail'},
    {'area':'daily','check':'selection_state_claim_is_proxy','observed':'target_state_proxy','status':'pass' if all('target_state_proxy' in report['execution_assumptions']['selection_state'] for report in daily_reports.values()) else 'fail'},
]
combined_validation = pd.DataFrame(validation_rows)
combined_validation.to_csv(OUTPUT / 'combined_validation.csv', index=False)
warnings = weekly_checks.query("status == 'warn'")
warnings.to_csv(OUTPUT / 'weekly_validation_warnings.csv', index=False)
print(combined_validation.to_string(index=False))
print('\nweekly warnings')
print(warnings.to_string(index=False))

selection_stats = daily_receipts.groupby(['window','arm','horizon'], as_index=False).agg(
    dates=('trade_date','size'), selected=('selected_count','mean'),
    retained=('retained_count','mean'), new=('new_position_count','mean'),
    target_cash=('cash_weight','mean'),
)
selection_stats.to_csv(OUTPUT / 'daily_selection_stats.csv', index=False)
print('\ndaily selection diagnostics')
print(selection_stats.to_string(index=False))

               area                                        check           observed status
weekly_reproduction                all_csv_numeric_outputs_match              18/18   pass
 weekly_reliability                  no_failed_validation_checks                  0   pass
          weekly_lp                        all_22_lp_checks_pass              22/22   pass
              daily              all_60_execution_cells_terminal                 60   pass
              daily               synthetic_capacity_never_bound                  0   pass
              daily target_entry_price_coverage_at_least_99.9pct           0.999596   pass
              daily               selection_state_claim_is_proxy target_state_proxy   pass

weekly warnings
                                      check status severity              observed                                                      expectation
  all_factor_missing_outcome_weight_bounded   warn  warning   0.17211538461538464                at most 10%

## Takeaways and next steps

1. **W-MultiHorizon 进入实现阶段。** 分别重训 D1、D2–5、D6–10、D11–20 增量收益。当前 decay 表只能证明方向值得做，不能替代重训结果。
2. **W-HardNeutral 保持 challenger。** 固定 neutral signal，候选池采用事前冻结的 60→100→200→500 扩池，或直接 Top500。不得静默放宽约束。主比较应报告 alpha 损失、tracking L1 与实际扩池分布。
3. **D-Stateful 先冻结 H1 与 H3。** H1 是最干净的状态效应。H3 是高潜力但跨窗不稳定的 challenger。H5 暂不优先。
4. **在下一批新数据前补 account-state。** 下一次选择必须读取实际 fills/holdings，并加入整手、容量/冲击、跨 cohort 净额与完整卖出可用量账本。
5. **不晋级 VD20。** 本轮固定使用 D 分数。慢成交量仍只作为后续低权重稳定器 shadow。

这些结果是研究优先级判断，不是交易建议。